# 17 — Full Improved Invoice Extraction Pipeline

Demo-ready notebook.  Completely self-contained — run top-to-bottom on a fresh
kernel with no other notebook having been run first.

Combines all 4 improvements into a single `process_invoice()` function.

| | |
|---|---|
| **Input** | Invoice image (jpg/png/webp/avif), PDF, or `PIL.Image` |
| **Output** | `dict` with 6 fields + confidence scores + validation notes |
| **Improvement 1** | Reading order sorting before LayoutLMv3 |
| **Improvement 2** | Confidence-gated extraction (threshold 0.70) |
| **Improvement 3** | DocTR OCR engine (Tesseract fallback if unavailable) |
| **Improvement 4** | Entity-level business rule validation + correction |
| **No retraining** | LayoutLMv3 weights read-only |
| **No generative AI** | All models are discriminative only |

**Workflow:**
1. Run Section 0 (setup) once after kernel restart — loads both models
2. Section 1 imports all improvement functions from `src/extraction_improvements.py`
3. Section 2 wraps everything into `process_invoice()`
4. Section 3 tests on 5 dataset examples with full output
5. Section 4 tests on your own images (edit `MY_IMAGES`)
6. Section 5 shows a before/after comparison table vs notebook 13

## 0. Setup — run after every kernel restart

In [22]:
import os, sys, time, subprocess

# Must be set before any transformers/tokenizers import
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'

t0 = time.time()
from transformers import (
    LayoutLMv3ForTokenClassification, LayoutLMv3Processor,
    AutoFeatureExtractor, AutoModelForImageClassification,
)
print('transformers imported in', round(time.time() - t0, 2), 'sec')
print('Python:', sys.executable)

transformers imported in 0.0 sec
Python: /Users/sofiiaavetisian/Desktop/UNI/statistical_learning/FINAL_PROJECT/.venv311/bin/python


In [23]:
import json
import torch
from pathlib import Path
from PIL import Image
from datasets import load_from_disk

PROJECT_ROOT  = Path('..').resolve()
DATASET_DIR   = PROJECT_ROOT / 'data' / 'processed' / 'layoutlmv3_dataset'
EXTRACTOR_CKPT = PROJECT_ROOT / 'models' / 'experimental' / 'layoutlmv3_fatura' / 'best_checkpoint'
FIGURES_DIR   = PROJECT_ROOT / 'reports' / 'figures'
MAX_LENGTH    = 512

# Document classifier checkpoint — used for type verification (optional)
# Set to None if not available; process_invoice() will skip type checking.
_CLASSIFIER_CKPT = PROJECT_ROOT / 'models' / 'experimental' / 'dit_fatura'
CLASSIFIER_CKPT  = _CLASSIFIER_CKPT if _CLASSIFIER_CKPT.exists() else None

DEVICE = torch.device(
    'cuda' if torch.cuda.is_available() else
    'mps'  if torch.backends.mps.is_available() else 'cpu'
)

with open(DATASET_DIR / 'label2id.json') as f:
    label2id = json.load(f)
with open(DATASET_DIR / 'id2label.json') as f:
    id2label = {int(k): v for k, v in json.load(f).items()}

FIELD_ORDER = [
    'INVOICE_NUMBER', 'INVOICE_DATE', 'DUE_DATE',
    'ISSUER_NAME', 'RECIPIENT_NAME', 'TOTAL_AMOUNT',
]
CLEAN_KEY = {
    'INVOICE_NUMBER': 'invoice_number', 'INVOICE_DATE': 'invoice_date',
    'DUE_DATE': 'due_date', 'ISSUER_NAME': 'issuer_name',
    'RECIPIENT_NAME': 'recipient_name', 'TOTAL_AMOUNT': 'total_amount',
}

raw_dataset = load_from_disk(str(DATASET_DIR))
print(f'Device          : {DEVICE}')
print(f'Extractor CKPT  : {EXTRACTOR_CKPT}')
print(f'Exists          : {EXTRACTOR_CKPT.exists()}')
print(f'Dataset         : {raw_dataset}')

Device          : mps
Extractor CKPT  : /Users/sofiiaavetisian/Desktop/UNI/statistical_learning/FINAL_PROJECT/document-classification/models/experimental/layoutlmv3_fatura/best_checkpoint
Exists          : True
Dataset         : DatasetDict({
    train: Dataset({
        features: ['image_path', 'words', 'bboxes', 'ner_tags'],
        num_rows: 1734
    })
    val: Dataset({
        features: ['image_path', 'words', 'bboxes', 'ner_tags'],
        num_rows: 371
    })
    test: Dataset({
        features: ['image_path', 'words', 'bboxes', 'ner_tags'],
        num_rows: 372
    })
})


In [24]:
# Load LayoutLMv3 field extractor
extractor_processor = LayoutLMv3Processor.from_pretrained(
    str(EXTRACTOR_CKPT),
    apply_ocr=False,
    use_fast=True,
    local_files_only=True,
)

extractor_model = LayoutLMv3ForTokenClassification.from_pretrained(
    str(EXTRACTOR_CKPT),
    id2label=id2label,
    label2id=label2id,
    local_files_only=True,
)
extractor_model.to(DEVICE)
extractor_model.eval()

print('Extractor loaded OK')
print('  tokenizer :', type(extractor_processor.tokenizer).__name__)
print('  num labels:', extractor_model.config.num_labels)
print('  device    :', DEVICE)

Loading weights: 100%|██████████| 216/216 [00:00<00:00, 8207.29it/s]


Extractor loaded OK
  tokenizer : LayoutLMv3Tokenizer
  num labels: 13
  device    : mps


In [25]:
# Load document classifier (optional — skip if checkpoint not available)
classifier_model     = None
classifier_processor = None

if CLASSIFIER_CKPT is not None:
    try:
        classifier_processor = AutoFeatureExtractor.from_pretrained(
            str(CLASSIFIER_CKPT), local_files_only=True
        )
        classifier_model = AutoModelForImageClassification.from_pretrained(
            str(CLASSIFIER_CKPT), local_files_only=True
        )
        classifier_model.to(DEVICE)
        classifier_model.eval()
        print(f'Classifier loaded OK  ({CLASSIFIER_CKPT.name})')
    except Exception as e:
        print(f'Classifier not loaded ({e.__class__.__name__}: {e})')
        print('process_invoice() will skip type verification.')
else:
    print('No classifier checkpoint found — type verification disabled.')

Classifier not loaded (OSError: Can't load feature extractor for '/Users/sofiiaavetisian/Desktop/UNI/statistical_learning/FINAL_PROJECT/document-classification/models/experimental/dit_fatura'. If you were trying to load it from 'https://huggingface.co/models', make sure you don't have a local directory with the same name. Otherwise, make sure '/Users/sofiiaavetisian/Desktop/UNI/statistical_learning/FINAL_PROJECT/document-classification/models/experimental/dit_fatura' is the correct path to a directory containing a preprocessor_config.json file)
process_invoice() will skip type verification.


## 1. Import All Improvement Functions

All 4 improvements are packaged in `src/extraction_improvements.py` so the
backend can import them directly without running any notebook.

In [26]:
SRC_DIR = str(PROJECT_ROOT / 'src')
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

import importlib
import extraction_improvements as ei

# Force reload so notebook picks up latest edits in src/extraction_improvements.py
# even if the module was imported earlier in this kernel session.
ei = importlib.reload(ei)

from invoice_cleaner import InvoiceCleaner

# Export names from reloaded module
sort_reading_order = ei.sort_reading_order
ROW_THRESHOLD = ei.ROW_THRESHOLD
FIELD_THRESHOLDS = getattr(ei, 'FIELD_THRESHOLDS', {
    'INVOICE_NUMBER': 0.70,
    'INVOICE_DATE':   0.70,
    'DUE_DATE':       0.70,
    'ISSUER_NAME':    0.95,
    'RECIPIENT_NAME': 0.50,
    'TOTAL_AMOUNT':   0.70,
})

get_raw_predictions_with_confidence = ei.get_raw_predictions_with_confidence
extract_with_confidence_gating = ei.extract_with_confidence_gating
ocr_image = ei.ocr_image
ocr_image_tesseract = ei.ocr_image_tesseract
ocr_image_doctr = ei.ocr_image_doctr
validate_and_correct_fields = ei.validate_and_correct_fields
_process_invoice_module = ei.process_invoice

cleaner = InvoiceCleaner()

print('All improvements loaded from src/extraction_improvements.py (reloaded)')
print(f'  Improvement 1 — sort_reading_order   (ROW_THRESHOLD={ROW_THRESHOLD})')
print('  Improvement 2 — extract_with_confidence_gating (per-field thresholds)')
print(f'    FIELD_THRESHOLDS={FIELD_THRESHOLDS}')
print('  Improvement 3 — ocr_image(engine="doctr") with Tesseract fallback')
print('  Improvement 4 — validate_and_correct_fields (5 business rules)')


All improvements loaded from src/extraction_improvements.py
  Improvement 1 — sort_reading_order   (ROW_THRESHOLD=15)
  Improvement 2 — extract_with_confidence_gating (threshold=0.70)
  Improvement 3 — ocr_image(engine="doctr") with Tesseract fallback
  Improvement 4 — validate_and_correct_fields (5 business rules)


## 2. Full Pipeline Function

`process_invoice()` is a thin wrapper around the module function that binds
the loaded models so callers only need to pass the image.

In [27]:
def _baseline_total_amount_only(image_or_path) -> str:
    """Notebook-13-style fallback for total_amount only."""
    import pytesseract

    # Load input as PIL image
    if isinstance(image_or_path, (str, Path)):
        image = Image.open(image_or_path).convert('RGB')
    elif isinstance(image_or_path, Image.Image):
        image = image_or_path.convert('RGB')
    else:
        image = Image.open(str(image_or_path)).convert('RGB')

    data = pytesseract.image_to_data(image, output_type=pytesseract.Output.DICT)
    words, boxes = [], []
    w, h = image.size

    for i, text in enumerate(data['text']):
        text = str(text).strip()
        if not text or int(data['conf'][i]) < 10:
            continue
        left, top = data['left'][i], data['top'][i]
        width, height = data['width'][i], data['height'][i]
        x0 = int(max(0, min(1000, round(left / w * 1000))))
        y0 = int(max(0, min(1000, round(top  / h * 1000))))
        x1 = int(max(0, min(1000, round((left + width)  / w * 1000))))
        y1 = int(max(0, min(1000, round((top  + height) / h * 1000))))
        if x1 <= x0:
            x1 = min(1000, x0 + 1)
        if y1 <= y0:
            y1 = min(1000, y0 + 1)
        words.append(text)
        boxes.append([x0, y0, x1, y1])

    words = words or ['empty']
    boxes = boxes or [[0, 0, 1, 1]]

    # Same token-to-field aggregation logic as notebook 13 baseline
    encoding = extractor_processor(
        image, words, boxes=boxes,
        truncation=True, padding='max_length',
        max_length=MAX_LENGTH, return_tensors='pt',
    )
    with torch.no_grad():
        outputs = extractor_model(**{k: v.to(DEVICE) for k, v in encoding.items()})

    token_preds = outputs.logits.argmax(-1).squeeze(0).cpu().tolist()
    word_ids = encoding.word_ids(batch_index=0)
    word_preds = {}
    for ti, wi in enumerate(word_ids):
        if wi is not None and wi not in word_preds:
            word_preds[wi] = token_preds[ti]

    aligned_words = [words[i] for i in sorted(word_preds)]
    aligned_pred_ids = [word_preds[i] for i in sorted(word_preds)]

    fields, current_field, current_tokens = {}, None, []
    for label_id, word in zip(aligned_pred_ids, aligned_words):
        label = id2label[label_id]
        if label == 'O':
            if current_field:
                t = ' '.join(current_tokens).strip()
                if t:
                    fields[current_field] = t
                current_field, current_tokens = None, []
        elif label.startswith('B-'):
            if current_field:
                t = ' '.join(current_tokens).strip()
                if t:
                    fields[current_field] = t
            current_field, current_tokens = label[2:], [word]
        elif label.startswith('I-'):
            fn = label[2:]
            if current_field == fn:
                current_tokens.append(word)
            elif current_field is None and fn in fields:
                fields[fn] += ' ' + word
            elif current_field is None:
                current_field, current_tokens = fn, [word]
            else:
                t = ' '.join(current_tokens).strip()
                if t:
                    fields[current_field] = t
                current_field, current_tokens = fn, [word]

    if current_field:
        t = ' '.join(current_tokens).strip()
        if t:
            fields[current_field] = t

    cleaned = cleaner.clean(fields, ocr_words=words)
    return cleaned.get('total_amount', '') or ''


def process_invoice(
    image_or_path,
    ocr_engine: str = 'doctr',
    confidence_threshold = FIELD_THRESHOLDS,
) -> dict:
    """
    Full improved invoice field extraction pipeline with baseline total_amount.

    Applies all improvements in the module, then overrides total_amount using
    notebook-13 baseline behavior to avoid the total_amount regressions observed
    in evaluation.
    """
    result = _process_invoice_module(
        image_or_path,
        extractor_model=extractor_model,
        extractor_processor=extractor_processor,
        device=DEVICE,
        id2label=id2label,
        cleaner=cleaner,
        classifier_model=classifier_model,
        classifier_processor=classifier_processor,
        ocr_engine=ocr_engine,
        confidence_threshold=confidence_threshold,
        max_length=MAX_LENGTH,
    )

    try:
        baseline_total = ''

        # Prefer the exact notebook-13 baseline function when available.
        if 'baseline_pipeline' in globals() and isinstance(image_or_path, (str, Path)):
            baseline_total = baseline_pipeline(str(image_or_path)).get('total_amount', '') or ''
        else:
            baseline_total = _baseline_total_amount_only(image_or_path)

        # Force baseline value, including empty string, to avoid regressions.
        result['total_amount'] = baseline_total
    except Exception as e:
        print(f'Baseline total_amount override failed: {e.__class__.__name__}: {e}')

    return result


def print_result(result: dict, title: str = '') -> None:
    """Pretty-print a process_invoice() result dict."""
    if title:
        print(f"\n{'='*70}")
        print(f'  {title}')
        print('='*70)
    for field_key in (
        'invoice_number', 'invoice_date', 'due_date',
        'issuer_name', 'recipient_name', 'total_amount',
    ):
        val = result.get(field_key, '') or '—'
        print(f"  {field_key:<20}: {val}")
    print(f"  {'source_mode':<20}: {result.get('source_mode', '—')}")
    confs = result.get('confidence_scores', {})
    if confs:
        print(f"  {'confidence_scores':<20}:")
        for fk, cv in confs.items():
            bar_len = int(cv * 20)
            bar = '█' * bar_len + '░' * (20 - bar_len)
            print(f"    {fk:<20}  [{bar}] {cv:.3f}")
    notes = result.get('validation_notes', [])
    if notes:
        print(f"  {'validation_notes':<20}:")
        for note in notes:
            print(f"    {note}")


print('process_invoice() and print_result() defined OK')


process_invoice() and print_result() defined OK


## 3. Test on Dataset Examples

Runs `process_invoice()` on 5 test-set examples from the FATURA dataset.
Shows full output including confidence scores and validation notes.

In [28]:
N_EXAMPLES = 5
SPLIT      = 'test'
OCR_ENGINE = 'doctr'      # change to 'tesseract' to compare
THRESHOLD  = FIELD_THRESHOLDS

for i in range(N_EXAMPLES):
    example    = raw_dataset[SPLIT][i]
    image_path = example['image_path']
    stem       = Path(image_path).stem

    result = process_invoice(
        image_path,
        ocr_engine=OCR_ENGINE,
        confidence_threshold=THRESHOLD,
    )
    print_result(result, title=f'{SPLIT.upper()} {i}: {stem}')


  TEST 0: Template1_Instance189
  invoice_number      : —
  invoice_date        : 29-Apr-2012
  due_date            : 07-Aug-2010
  issuer_name         : —
  recipient_name      : Shelly Rodriguez Markville AK
  total_amount        : 134.41 USD
  source_mode         : image_ocr
  confidence_scores   :
    INVOICE_NUMBER        [░░░░░░░░░░░░░░░░░░░░] 0.000
    INVOICE_DATE          [███████████████████░] 0.998
    DUE_DATE              [███████████████████░] 0.997
    ISSUER_NAME           [░░░░░░░░░░░░░░░░░░░░] 0.000
    RECIPIENT_NAME        [░░░░░░░░░░░░░░░░░░░░] 0.000
    TOTAL_AMOUNT          [███████████████████░] 0.998

  TEST 1: Template38_Instance29
  invoice_number      : 9Y1M9d-232
  invoice_date        : 15-Feb-1993
  due_date            : 14-Jan-2007
  issuer_name         : Lake Antonioshire
  recipient_name      : —
  total_amount        : 220.90 EUR
  source_mode         : image_ocr
  confidence_scores   :
    INVOICE_NUMBER        [███████████████████░] 0.993
    INVOIC

## 4. Test on Your Own Images

Add paths to real invoices in `MY_IMAGES` and run.  Supports jpg, png, webp,
avif, and PDF.  Native-text PDFs are rasterised for LayoutLMv3.

In [29]:
MY_IMAGES = [
    '/Users/sofiiaavetisian/Desktop/UNI/statistical_learning/FINAL_PROJECT/document-classification/data/external/doc_i_2.avif',
    '/Users/sofiiaavetisian/Desktop/UNI/statistical_learning/FINAL_PROJECT/document-classification/data/external/doc_i_1.webp',
    '/Users/sofiiaavetisian/Desktop/UNI/statistical_learning/FINAL_PROJECT/document-classification/data/external/doc_i_3.webp',
    '/Users/sofiiaavetisian/Desktop/UNI/statistical_learning/FINAL_PROJECT/document-classification/data/external/invoice-0-4.pdf',
    '/Users/sofiiaavetisian/Desktop/UNI/statistical_learning/FINAL_PROJECT/document-classification/data/external/invoice-1-3.pdf',
    '/Users/sofiiaavetisian/Desktop/UNI/statistical_learning/FINAL_PROJECT/document-classification/data/external/invoice-2-1.pdf',
    '/Users/sofiiaavetisian/Desktop/UNI/statistical_learning/FINAL_PROJECT/document-classification/data/external/invoice-3-0.pdf',
    '/Users/sofiiaavetisian/Desktop/UNI/statistical_learning/FINAL_PROJECT/document-classification/data/external/invoice-7-0.pdf',
]

for img_path in MY_IMAGES:
    p = Path(img_path)
    if not p.exists():
        print(f'\n  File not found: {p.name} — skipping')
        continue
    try:
        result = process_invoice(str(p), ocr_engine='doctr', confidence_threshold=FIELD_THRESHOLDS)
        print_result(result, title=p.name)
    except Exception as e:
        print(f'\n  Error processing {p.name}: {e.__class__.__name__}: {e}')


  doc_i_2.avif
  invoice_number      : —
  invoice_date        : —
  due_date            : —
  issuer_name         : Ketut Susilo
  recipient_name      : —
  total_amount        : —
  source_mode         : image_ocr
  confidence_scores   :
    INVOICE_NUMBER        [██████████████████░░] 0.926
    INVOICE_DATE          [░░░░░░░░░░░░░░░░░░░░] 0.000
    DUE_DATE              [██████████████████░░] 0.914
    ISSUER_NAME           [██████████████████░░] 0.933
    RECIPIENT_NAME        [░░░░░░░░░░░░░░░░░░░░] 0.000
    TOTAL_AMOUNT          [███████████████████░] 0.975

  doc_i_1.webp
  invoice_number      : 12345
  invoice_date        : —
  due_date            : —
  issuer_name         : Imani Olowe
  recipient_name      : Imani Olowe +123-456-7890
  total_amount        : $2750
  source_mode         : image_ocr
  confidence_scores   :
    INVOICE_NUMBER        [███████████████████░] 0.998
    INVOICE_DATE          [█████████████████░░░] 0.886
    DUE_DATE              [░░░░░░░░░░░░░░░░░░░░

## 5. Before / After Comparison vs Notebook 13

Same 5 test-set examples through the original notebook 13 pipeline (no sorting,
no confidence gating, no validation) versus the full improved pipeline.

Fields that changed are marked with `<<`.

In [30]:
"""Baseline pipeline — identical logic to notebook 13, no improvements applied."""
import torch.nn.functional as F

def _baseline_get_raw(image, words, bboxes):
    encoding = extractor_processor(
        image, words, boxes=bboxes,
        truncation=True, padding='max_length',
        max_length=MAX_LENGTH, return_tensors='pt',
    )
    with torch.no_grad():
        outputs = extractor_model(**{k: v.to(DEVICE) for k, v in encoding.items()})
    token_preds = outputs.logits.argmax(-1).squeeze(0).cpu().tolist()
    word_ids    = encoding.word_ids(batch_index=0)
    word_preds  = {}
    for ti, wi in enumerate(word_ids):
        if wi is not None and wi not in word_preds:
            word_preds[wi] = token_preds[ti]
    aligned_words    = [words[i]      for i in sorted(word_preds)]
    aligned_pred_ids = [word_preds[i] for i in sorted(word_preds)]
    fields, current_field, current_tokens = {}, None, []
    for label_id, word in zip(aligned_pred_ids, aligned_words):
        label = id2label[label_id]
        if label == 'O':
            if current_field:
                t = ' '.join(current_tokens).strip()
                if t: fields[current_field] = t
                current_field, current_tokens = None, []
        elif label.startswith('B-'):
            if current_field:
                t = ' '.join(current_tokens).strip()
                if t: fields[current_field] = t
            current_field, current_tokens = label[2:], [word]
        elif label.startswith('I-'):
            fn = label[2:]
            if current_field == fn:
                current_tokens.append(word)
            elif current_field is None and fn in fields:
                fields[fn] += ' ' + word
            elif current_field is None:
                current_field, current_tokens = fn, [word]
            else:
                t = ' '.join(current_tokens).strip()
                if t: fields[current_field] = t
                current_field, current_tokens = fn, [word]
    if current_field:
        t = ' '.join(current_tokens).strip()
        if t: fields[current_field] = t
    return fields


def baseline_pipeline(image_path: str) -> dict:
    """Notebook 13 pipeline: Tesseract OCR → raw predictions → InvoiceCleaner."""
    import pytesseract
    image = Image.open(image_path).convert('RGB')
    data  = pytesseract.image_to_data(image, output_type=pytesseract.Output.DICT)
    words, boxes = [], []
    w, h = image.size
    for i, text in enumerate(data['text']):
        text = str(text).strip()
        if not text or int(data['conf'][i]) < 10:
            continue
        left, top = data['left'][i], data['top'][i]
        width, height = data['width'][i], data['height'][i]
        x0 = int(max(0, min(1000, round(left / w * 1000))))
        y0 = int(max(0, min(1000, round(top  / h * 1000))))
        x1 = int(max(0, min(1000, round((left + width)  / w * 1000))))
        y1 = int(max(0, min(1000, round((top  + height) / h * 1000))))
        if x1 <= x0: x1 = min(1000, x0 + 1)
        if y1 <= y0: y1 = min(1000, y0 + 1)
        words.append(text)
        boxes.append([x0, y0, x1, y1])
    words = words or ['empty']
    boxes = boxes or [[0, 0, 1, 1]]
    raw     = _baseline_get_raw(image, words, boxes)
    cleaned = cleaner.clean(raw, ocr_words=words)
    return cleaned


print('Baseline pipeline function defined OK')

Baseline pipeline function defined OK


In [32]:
N_COMPARE = 100
SPLIT     = 'test'

total_changed = 0
total_fields  = 0

for i in range(N_COMPARE):
    example = raw_dataset[SPLIT][i]
    img_path = example['image_path']
    stem     = Path(img_path).stem

    # Baseline (notebook 13)
    try:
        old_result = baseline_pipeline(img_path)
    except Exception as e:
        print(f'Baseline failed for example {i}: {e}')
        old_result = {}

    # Improved pipeline
    new_result = process_invoice(img_path, ocr_engine='doctr', confidence_threshold=FIELD_THRESHOLDS)

    print(f"\n{'='*85}")
    print(f"  {SPLIT.upper()} {i}: {stem}")
    print('='*85)
    print(f"  {'FIELD':<20}  {'NOTEBOOK 13 (BASELINE)':^28}  IMPROVED  ")
    print(f"  {'-'*20}  {'-'*28}  {'-'*28}")

    for field in FIELD_ORDER:
        ck       = CLEAN_KEY[field]
        old_val  = old_result.get(ck, '') or '—'
        new_val  = new_result.get(ck, '') or '—'
        changed  = old_val != new_val
        marker   = ' <<' if changed else ''
        old_d    = (old_val[:26] + '..') if len(old_val) > 28 else old_val
        new_d    = (new_val[:26] + '..') if len(new_val) > 28 else new_val
        print(f"  {field:<20}  {old_d:<28}  {new_d}{marker}")
        total_fields  += 1
        total_changed += int(changed)

    notes = new_result.get('validation_notes', [])
    if notes:
        print(f"  Validation notes:")
        for note in notes:
            print(f"    {note}")

print(f"\n{'='*85}")
print(f"  Summary: {total_changed}/{total_fields} fields changed across {N_COMPARE} examples")
pct = 100 * total_changed / total_fields if total_fields else 0
print(f"  Change rate: {pct:.0f}%")
print(f"  << = field value differs between baseline and improved pipeline")


  TEST 0: Template1_Instance189
  FIELD                    NOTEBOOK 13 (BASELINE)     IMPROVED  
  --------------------  ----------------------------  ----------------------------
  INVOICE_NUMBER        —                             —
  INVOICE_DATE          29-Apr-2012                   29-Apr-2012
  DUE_DATE              07-Aug-2010                   07-Aug-2010
  ISSUER_NAME           Mission                       — <<
  RECIPIENT_NAME        Shelly Rodriguez              Shelly Rodriguez Markville.. <<
  TOTAL_AMOUNT          134.41 USD                    134.41 USD

  TEST 1: Template38_Instance29
  FIELD                    NOTEBOOK 13 (BASELINE)     IMPROVED  
  --------------------  ----------------------------  ----------------------------
  INVOICE_NUMBER        1M9d-232                      9Y1M9d-232 <<
  INVOICE_DATE          15-Feb-1993                   15-Feb-1993
  DUE_DATE              —                             14-Jan-2007 <<
  ISSUER_NAME           Lake Antonios